In [ ]:
from discovery_utils.utils import search
from discovery_utils.getters import gtr

from discovery_utils import PROJECT_DIR
VECTOR_DB_DIR = PROJECT_DIR / 'tmp/vector_db'

GTR = gtr.GtrGetter(vector_db_path=VECTOR_DB_DIR)

In [ ]:
Search = search.SearchDataset(GTR, GTR.projects_enriched, "config.yaml")

In [ ]:
search_df = Search.do_search()

In [87]:
from discovery_utils.utils import (
    analysis,
    analysis_gtr,
)
import importlib
importlib.reload(analysis);
importlib.reload(analysis_gtr);

In [ ]:
df = (
    search_df
    .merge(GTR.get_projects_text(), on='id', how='left')
    .query("_score_avg > 0.3")
)
print(len(df))
df_dedup = analysis_gtr.deduplicate_projects(df, description_column='text')
print(len(df_dedup))

In [ ]:
analysis_gtr.funding_per_period(df, period='year', min_year=2010, max_year=2024)

In [ ]:
ts_df = analysis_gtr.get_timeseries(df, period='year', min_year=2010, max_year=2024)
analysis.magnitude_growth(ts_df, 2019, 2024)
# ts_df

In [ ]:
from discovery_utils.getters import crunchbase
CB = crunchbase.CrunchbaseGetter(vector_db_path=VECTOR_DB_DIR)

In [ ]:
SearchCB = search.SearchDataset(CB, CB.organisations_enriched, "config.yaml")
search_cb_df = SearchCB.do_search()

In [ ]:
orgs_df = CB.organisations_enriched.copy()
funds_df = CB.funding_rounds_enriched.copy()

In [ ]:
importlib.reload(crunchbase);
CB = crunchbase.CrunchbaseGetter(vector_db_path=VECTOR_DB_DIR)
CB._organisations_enriched = orgs_df
CB._funding_rounds_enriched = funds_df

In [116]:
df = (
    search_cb_df
    .query("_score_avg > 0.3")
)

In [204]:
from discovery_utils.utils import (
    analysis,
    analysis_crunchbase,
)
importlib.reload(analysis);
importlib.reload(analysis_crunchbase);

In [ ]:
analysis_crunchbase.orgs_founded_per_period(df, 'year', 2010, 2024)

In [133]:
matching_ids = df.id.to_list()

In [134]:
selected_funding_df = (
    CB.funding_rounds_enriched
    .query("org_id in @matching_ids")
    .query(f"year >= {2010}")
    .query(f"year <= {2024}")
    # .query(f"investment_type in @include_deals")
    .drop_duplicates("funding_round_id")
)

In [191]:
importlib.reload(analysis_crunchbase);

In [ ]:
analysis_crunchbase

In [ ]:
deals_df, deal_counts_df = analysis_crunchbase.get_funding_by_year_and_range(selected_funding_df, 2014, 2024)
deals_df

In [ ]:
deal_counts_df

In [ ]:
(
    selected_funding_df
    .query("announced_on >= '2015-01-01'")
    .query("announced_on < '2016-01-01'")
    .raised_amount_gbp.sum()
)

In [276]:
importlib.reload(analysis_crunchbase);
ts_df = analysis_crunchbase.get_timeseries(df, selected_funding_df, 'year', 2010, 2024)

In [ ]:
ts_df

In [ ]:
analysis.magnitude_growth(ts_df, 2019, 2024)

In [285]:
importlib.reload(analysis_crunchbase);

In [ ]:
aggregated_funding_types_df = analysis_crunchbase.aggregate_by_funding_round_types(selected_funding_df)
analysis_crunchbase.chart_investment_types(aggregated_funding_types_df)

In [ ]:
analysis_crunchbase.chart_deal_sizes(deals_df)

In [ ]:
analysis_crunchbase.chart_deal_sizes_counts(deal_counts_df)

In [320]:
from discovery_utils.utils import charts
importlib.reload(charts);


In [ ]:
ts_df.head(5)

In [ ]:
charts.ts_bar_incomplete(
    ts_df,
    variable='raised_amount_gbp_total',
    variable_title='Funding',
    max_complete_year=2022
)

In [330]:
orgs_to_narrow_categories_df = CB.organisation_categories.explode('category_list')

In [ ]:
orgs_to_narrow_categories_df.head(5)

In [333]:
_df = (
    orgs_to_narrow_categories_df
    .merge(CB.group_to_categories, left_on='category_list', right_on='category')
)

In [ ]:
CB.group_to_categories

In [ ]:
orgs_to_narrow_categories_df.merge()

In [349]:
df = CB.get_companies_in_categories(['Family'])


In [ ]:
CB.unique_funding_round_types

In [352]:
funding_rounds_df = CB.select_funding_rounds(
    org_ids=df.id.to_list(),
    funding_round_types=['angel', 'seed', 'pre_seed', 'series_a']
)

In [356]:
importlib.reload(analysis_crunchbase);

In [358]:
ts_df = analysis_crunchbase.get_timeseries(df, funding_rounds_df, 'year', 2014, 2024)

In [ ]:
charts.ts_bar(
    ts_df,
    "raised_amount_gbp_total",
    "Funding",
)

In [ ]:
CB.group_to_categories.category.unique()

In [ ]:
CB.group_vectors

In [ ]:
CB.find_similar_categories("cyber safety", category_type='narrow')

In [ ]:
unique_groups = CB.group_to_categories.group.unique()
vectors = CB.embedding_model.encode(unique_groups)

In [ ]:
len(unique_groups)

In [ ]:
len(list(vectors))

In [379]:
import pandas as pd
vectors_df = pd.DataFrame(
    data={"group": unique_groups, "vector": list(vectors)}
)

In [ ]:
query_embedding = CB.embedding_model.encode(["Cybersecurity"])[0]

In [ ]:
import numpy as np
(
    vectors_df            
    .assign(similarity = vectors_df.vector.apply(lambda x: np.dot(query_embedding, x)))
    .sort_values("similarity", ascending=False)
    .drop(columns="vector")
    .head(10)
)

In [396]:
orgs_cyber_df = CB.get_companies_in_categories(["Cyber Security"])

In [400]:
orgs_parenting_df = CB.get_companies_in_categories(["Family", "Parenting", "Children", "Teenagers", "Child Care"])

In [401]:
matching_ids = set(orgs_cyber_df.id.to_list()).intersection(orgs_parenting_df.id.to_list())

In [ ]:
len(matching_ids)

In [ ]:
importlib.reload(charts)

In [404]:
# CB.organisations_enriched.query("id in @matching_ids")

In [417]:
ts_test = pd.concat([
    ts_df.assign(category="Cyber Security"),
    ts_df.assign(category="Parenting").assign(raised_amount_gbp_total=lambda df: df.raised_amount_gbp_total * 0.5)
], ignore_index=True)

In [ ]:
(
    charts.configure_plots(
        charts.ts_bar_incomplete(
        ts_test,
        "raised_amount_gbp_total",
        "Funding",
        categories_to_show=["Cyber Security", "Parenting"],
        category_column="category",
        max_complete_year=2022
    ), "Ahahaha")
)


